# Model Output Analysis

Comparing **system prompt** and **temperature** impact on model behavior.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

df = pd.read_csv("experiements_model_single_turn.csv")

# Data summary
temp_counts = df['temperature'].value_counts(dropna=False).sort_index()
n_temps = len(temp_counts)

print(f"Total distinct prompts: {len(df)}")
print(f"Distinct temperatures : {list(temp_counts.to_dict().keys())}")
print(f"\nRequests per temperature:")
for t, count in temp_counts.items():
    label = "Default" if pd.isna(t) else f"T={t}"
    print(f"  - {label}: {count} ({100*count/len(df):.1f}%)")


df.head(2)

Total distinct prompts: 1425
Distinct temperatures : [0.3, 0.7, nan]

Requests per temperature:
  - T=0.3: 440 (30.9%)
  - T=0.7: 600 (42.1%)
  - Default: 385 (27.0%)


,timestamp,prompt,expected_tool,mismatch,finish_reason,completion_tokens,has_system_prompt,temperature,tool_choice,has_reasoning,has_content,has_tool,raw_analysis,parsed_reasoning,raw_final,parsed_content,raw_tool,parsed_tool,raw_output
0,2026-02-02T22:54:45.627301,What is TEE?...,1,False,tool_calls,107,False,NaN,auto,True,False,True,"The user asks: ""What is TEE?"" They want explan...","The user asks: ""What is TEE?"" They want explan...",NaN,NaN,"get_confidential_computing_info({ ""file"": ""c...","get_confidential_computing_info({""file"": ""conf...",NaN
1,2026-02-02T22:55:28.037306,What is TEE?...,1,False,tool_calls,123,False,NaN,auto,True,False,True,We need to provide explanation of Trusted Exec...,We need to provide explanation of Trusted Exec...,NaN,NaN,"get_confidential_computing_info({ ""file"": ""c...","get_confidential_computing_info({""file"": ""conf...",NaN


---
## 0. vLLM Parsing Check

Comparing raw model output vs vLLM parsed output to detect parsing errors.

In [ ]:
# Check if raw and parsed exist together for each field
n = len(df)

pairs = [
    ('Reasoning', 'raw_analysis', 'parsed_reasoning'),
    ('Content', 'raw_final', 'parsed_content'),
    ('Tool', 'raw_tool', 'parsed_tool'),
]

print(f"Total: {n} requests\n")

for label, raw_col, parsed_col in pairs:
    raw_exists = df[raw_col].notna()
    parsed_exists = df[parsed_col].notna()
    match = (raw_exists == parsed_exists).sum()
    mismatch = n - match

    emoji = "🟢" if mismatch == 0 else ("🟠" if mismatch <= n*0.01 else "🔴")
    print(f"{emoji} {label}: {match}/{n} match ({mismatch} mismatch)")

Total: 1425 requests

🟢 Reasoning: 1425/1425 match (0 mismatch)
🟢 Content: 1425/1425 match (0 mismatch)
🟢 Tool: 1425/1425 match (0 mismatch)


In [ ]:
# Helper functions
def stats(d):
    n = len(d)
    if n == 0: return None
    return {
        'n': n,
        'reasoning': int(d['parsed_reasoning'].notna().sum()),
        'tool': int(d['parsed_tool'].notna().sum()),
        'content': int(d['parsed_content'].notna().sum()),
        'mismatch': int(d['mismatch'].sum())
    }

def flt(d, sys=None, t=None):
    r = d.copy()
    if sys is not None: r = r[r['has_system_prompt'] == sys]
    if t == 'def': r = r[r['temperature'].isna()]
    elif t is not None: r = r[r['temperature'] == t]
    return r

def fmt(s, key, with_emoji=False):
    if not s: return "—"
    val = s[key]
    pct = 100 * val / s['n']
    base = f"{val}/{s['n']} ({pct:.0f}%)"
    if not with_emoji:
        return base
    # Emoji for mismatch: green=0%, orange=1-10%, red=>10%
    if pct == 0:
        return f"🟢 {base}"
    elif pct <= 10:
        return f"🟠 {base}"
    else:
        return f"🔴 {base}"

# Auto-detect temperatures from data
temp_values = sorted([t for t in df['temperature'].dropna().unique()])
temps = [('Default', 'def')] + [(f'T={t}', t) for t in temp_values]

# Build all configurations (sys_prompt × temperature)
configs = []
for t_label, t_val in temps:
    configs.append((f'No Sys {t_label}' if t_label != 'Default' else 'No Sys', False, t_val if t_label != 'Default' else 'def'))
for t_label, t_val in temps:
    configs.append((f'Sys {t_label}' if t_label != 'Default' else 'Sys', True, t_val if t_label != 'Default' else 'def'))

---
## 1. Stats per Prompt

In [ ]:
# Build header
header = "| Prompt |" + " | ".join([c[0] for c in configs]) + " |\n"
header += "|--------|" + "|".join(["-------" for _ in configs]) + "|\n"

# Build rows with mismatch totals for sorting
prompt_data = []
for prompt in df['prompt'].dropna().unique():
    pdf = df[df['prompt'] == prompt]
    total_mismatch = pdf['mismatch'].sum()
    row = f"| {prompt[:60]}... |" if len(prompt) > 60 else f"| {prompt} |"
    for _, sys, t in configs:
        s = stats(flt(pdf, sys, t))
        row += f" {fmt(s, 'mismatch', with_emoji=True)} |"
    prompt_data.append((total_mismatch, row))

# Sort by mismatch count (descending)
rows = [row for _, row in sorted(prompt_data, key=lambda x: -x[0])]

display(Markdown(header + "\n".join(rows)))

| Prompt |No Sys | No Sys T=0.3 | No Sys T=0.7 | Sys | Sys T=0.3 | Sys T=0.7 |
|--------|-------|-------|-------|-------|-------|-------|
| What is TEE?... | 🔴 7/35 (20%) | 🔴 5/10 (50%) | 🔴 11/40 (28%) | 🟢 0/10 (0%) | 🟢 0/40 (0%) | 🟢 0/40 (0%) |
| Write a Python script that prints hello to /tmp/he... | 🔴 4/10 (40%) | 🔴 5/10 (50%) | 🔴 8/20 (40%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| What is TDX?... | 🟠 1/20 (5%) | 🟠 1/10 (10%) | 🟠 1/20 (5%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| What is Confidential Compute?... | 🟢 0/10 (0%) | 🟢 0/10 (0%) | 🟠 1/20 (5%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| List the files in /tmp... | 🟢 0/10 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| What's my current directory?... | 🟢 0/10 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| Show running processes... | 🟢 0/10 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| Read all the Python files in cvm/ and summarize th... | 🟢 0/60 (0%) | 🟢 0/20 (0%) | 🟢 0/40 (0%) | 🟢 0/30 (0%) | 🟢 0/40 (0%) | 🟢 0/40 (0%) |
| Create a file /tmp/test.txt with 'hello world' ... | 🟢 0/10 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| Run pwd... | 🟢 0/10 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| Check disk usage with df -h... | 🟢 0/10 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| Find all .py files in the current directory... | 🟢 0/10 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |
| List all markdown files... | 🟢 0/30 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/10 (0%) | 🟢 0/20 (0%) | 🟢 0/20 (0%) |

---
## 2. Global Stats (All Configurations)

In [ ]:
header = "| Metric | " + " | ".join([c[0] for c in configs]) + " |\n"
header += "|--------|" + "|".join(["-------" for _ in configs]) + "|\n"

rows = []
for metric, key in [('N', 'n'), ('Reasoning', 'reasoning'), ('Tool', 'tool'), ('Content', 'content'), ('Mismatch', 'mismatch')]:
    row = f"| **{metric}** |"
    for _, sys, t in configs:
        s = stats(flt(df, sys, t))
        if key == 'n': row += f" {s['n'] if s else '—'} |"
        elif key == 'mismatch': row += f" {fmt(s, key, with_emoji=True)} |"
        else: row += f" {fmt(s, key)} |"
    rows.append(row)

display(Markdown(header + "\n".join(rows)))

| Metric | No Sys | No Sys T=0.3 | No Sys T=0.7 | Sys | Sys T=0.3 | Sys T=0.7 |
|--------|-------|-------|-------|-------|-------|-------|
| **N** | 235 | 140 | 300 | 150 | 300 | 300 |
| **Reasoning** | 235/235 (100%) | 140/140 (100%) | 300/300 (100%) | 150/150 (100%) | 300/300 (100%) | 300/300 (100%) |
| **Tool** | 223/235 (95%) | 129/140 (92%) | 279/300 (93%) | 150/150 (100%) | 300/300 (100%) | 300/300 (100%) |
| **Content** | 12/235 (5%) | 11/140 (8%) | 21/300 (7%) | 0/150 (0%) | 0/300 (0%) | 0/300 (0%) |
| **Mismatch** | 🟠 12/235 (5%) | 🟠 11/140 (8%) | 🟠 21/300 (7%) | 🟢 0/150 (0%) | 🟢 0/300 (0%) | 🟢 0/300 (0%) |

---
## 3. System Prompt Only

In [ ]:
sys_configs = [('Without Sys Prompt', False), ('With Sys Prompt', True)]
header = "| Metric | " + " | ".join([c[0] for c in sys_configs]) + " |\n"
header += "|--------|" + "|".join(["-------" for _ in sys_configs]) + "|\n"

rows = []
for metric, key in [('N', 'n'), ('Reasoning', 'reasoning'), ('Tool', 'tool'), ('Content', 'content'), ('Mismatch', 'mismatch')]:
    row = f"| **{metric}** |"
    for _, sys in sys_configs:
        s = stats(flt(df, sys=sys))
        if key == 'n': row += f" {s['n'] if s else '—'} |"
        elif key == 'mismatch': row += f" {fmt(s, key, with_emoji=True)} |"
        else: row += f" {fmt(s, key)} |"
    rows.append(row)

display(Markdown(header + "\n".join(rows)))

| Metric | Without Sys Prompt | With Sys Prompt |
|--------|-------|-------|
| **N** | 675 | 750 |
| **Reasoning** | 675/675 (100%) | 750/750 (100%) |
| **Tool** | 631/675 (93%) | 750/750 (100%) |
| **Content** | 44/675 (7%) | 0/750 (0%) |
| **Mismatch** | 🟠 44/675 (7%) | 🟢 0/750 (0%) |

---
## 4. Temperature Only

In [ ]:
header = "| Metric | " + " | ".join([t[0] for t in temps]) + " |\n"
header += "|--------|" + "|".join(["-------" for _ in temps]) + "|\n"

rows = []
for metric, key in [('N', 'n'), ('Reasoning', 'reasoning'), ('Tool', 'tool'), ('Content', 'content'), ('Mismatch', 'mismatch')]:
    row = f"| **{metric}** |"
    for _, t in temps:
        s = stats(flt(df, t=t))
        if key == 'n': row += f" {s['n'] if s else '—'} |"
        elif key == 'mismatch': row += f" {fmt(s, key, with_emoji=True)} |"
        else: row += f" {fmt(s, key)} |"
    rows.append(row)

display(Markdown(header + "\n".join(rows)))

| Metric | Default | T=0.3 | T=0.7 |
|--------|-------|-------|-------|
| **N** | 385 | 440 | 600 |
| **Reasoning** | 385/385 (100%) | 440/440 (100%) | 600/600 (100%) |
| **Tool** | 373/385 (97%) | 429/440 (98%) | 579/600 (96%) |
| **Content** | 12/385 (3%) | 11/440 (2%) | 21/600 (4%) |
| **Mismatch** | 🟠 12/385 (3%) | 🟠 11/440 (2%) | 🟠 21/600 (4%) |

---
## 5. Conclusion

In [ ]:
no_sys = stats(flt(df, sys=False))
with_sys = stats(flt(df, sys=True))

if no_sys and with_sys:
    # Breakdown by temperature for each system prompt config
    print("Formula:")
    print("  mismatch_rate = Σ(mismatch per temp) / Σ(n per temp)")
    print()

    for label, sys_val in [("Without sys prompt", False), ("With sys prompt", True)]:
        parts_num = []
        parts_den = []
        for t_label, t_val in temps:
            s = stats(flt(df, sys=sys_val, t=t_val))
            if s:
                parts_num.append(f"{s['mismatch']}")
                parts_den.append(f"{s['n']}")

        s_total = stats(flt(df, sys=sys_val))
        rate = 100 * s_total['mismatch'] / s_total['n']

        print(f"{label}:")
        print(f"  = ({' + '.join(parts_num)}) / ({' + '.join(parts_den)})")
        print(f"  = {s_total['mismatch']} / {s_total['n']}")
        print(f"  = {rate:.1f}%")
        print()

    rate_no_sys = 100 * no_sys['mismatch'] / no_sys['n']
    rate_with_sys = 100 * with_sys['mismatch'] / with_sys['n']
    diff = rate_no_sys - rate_with_sys

    print(f"Difference: {rate_no_sys:.1f}% - {rate_with_sys:.1f}% = {diff:.1f}%")
    print()

    if diff > 0:
        print(f"✅ System prompt reduces mismatch by {diff:.1f}%")
    elif diff < 0:
        print(f"❌ System prompt increases mismatch by {abs(diff):.1f}%")
    else:
        print("✅ No difference")
else:
    print("Need more data")

Formula:
  mismatch_rate = Σ(mismatch per temp) / Σ(n per temp)

Without sys prompt:
  = (12 + 11 + 21) / (235 + 140 + 300)
  = 44 / 675
  = 6.5%

With sys prompt:
  = (0 + 0 + 0) / (150 + 300 + 300)
  = 0 / 750
  = 0.0%

Difference: 6.5% - 0.0% = 6.5%

✅ System prompt reduces mismatch by 6.5%


 History:

```
  python check_model_output_single_turn.py "Hello, how are you?" 0 --runs 20                          
  python check_model_output_single_turn.py "Hello, how are you?" 0 --runs 20 -t 0.3                      
  python check_model_output_single_turn.py "Hello, how are you?" 0 --runs 20 -t 0.7                      
  python check_model_output_single_turn.py "Hello, how are you?" 0 --runs 20 -s -t 0.3                            
  python check_model_output_single_turn.py "Hello, how are you?" 0 --runs 20 -s -t 0.7

  python check_model_output_single_turn.py "What is 2+2?" 0 --runs 20                          
  python check_model_output_single_turn.py "What is 2+2?" 0 --runs 20 -t 0.3                      
  python check_model_output_single_turn.py "What is 2+2?" 0 --runs 20 -t 0.7                      
  python check_model_output_single_turn.py "What is 2+2?" 0 --runs 20 -s -t 0.3                            
  python check_model_output_single_turn.py "What is 2+2?" 0 --runs 20 -s -t 0.7

  python check_model_output_single_turn.py "What is the capital of France?" 0 --runs 20                          
  python check_model_output_single_turn.py "What is the capital of France?" 0 --runs 20 -t 0.3                      
  python check_model_output_single_turn.py "What is the capital of France?" 0 --runs 20 -t 0.7                      
  python check_model_output_single_turn.py "What is the capital of France?" 0 --runs 20 -s -t 0.3                            
  python check_model_output_single_turn.py "What is the capital of France?" 0 --runs 20 -s -t 0.7


  python check_model_output_single_turn.py "Hello, how are you?" 0 --runs 20 -s -n 5                             
  python check_model_output_single_turn.py "What is 2+2?" 0 --runs 20 -s -n 5                                      
  python check_model_output_single_turn.py "Write a haiku about the moon" 0 --runs 20 -s -n 5
  python check_model_output_single_turn.py "What is the capital of France?" 0 --runs 20 -s -n 5



python check_model_output_single_turn.py "What is TEE?" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "List the files in /tmp" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "What's my current directory?" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "Show running processes" 1  --runs 20 -t 0.7
python check_model_output_single_turn.py "Read all the Python files in cvm/ and summarize them" 1  --runs 20 -t 0.7
python check_model_output_single_turn.py "Create a file /tmp/test.txt with 'hello world' " 1 --runs 20 -t 0.7

python check_model_output_single_turn.py "Run pwd" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "Check disk usage with df -h" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "Find all .py files in the current directory" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "Write a Python script that prints hello to /tmp/hello.py" 1 --runs 20 -t 0.7

python check_model_output_single_turn.py "List all markdown files" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "What is TEE?" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "What is Confidential Compute?" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "What is TDX?" 1 --runs 20 -t 0.7
python check_model_output_single_turn.py "Read all the Python files in cvm/ and summarize them" 1 --runs 20 -t 0.7

python check_model_output_single_turn.py "What is TEE?" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "List the files in /tmp" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "What's my current directory?" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "Show running processes" 1  --runs 20 -s -t 0.7
python check_model_output_single_turn.py "Read all the Python files in cvm/ and summarize them" 1  --runs 20 -s -t 0.7
python check_model_output_single_turn.py "Create a file /tmp/test.txt with 'hello world' " 1 --runs 20 -s -t 0.7

python check_model_output_single_turn.py "Run pwd" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "Check disk usage with df -h" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "Find all .py files in the current directory" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "Write a Python script that prints hello to /tmp/hello.py" 1 --runs 20 -s -t 0.7

python check_model_output_single_turn.py "List all markdown files" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "What is TEE?" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "What is Confidential Compute?" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "What is TDX?" 1 --runs 20 -s -t 0.7
python check_model_output_single_turn.py "Read all the Python files in cvm/ and summarize them" 1 --runs 20 -s -t 0.7

python check_model_output_single_turn.py "What is TEE?" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "List the files in /tmp" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "What's my current directory?" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "Show running processes" 1  --runs 20 -s -t 0.3
python check_model_output_single_turn.py "Read all the Python files in cvm/ and summarize them" 1  --runs 20 -s -t 0.3
python check_model_output_single_turn.py "Create a file /tmp/test.txt with 'hello world' " 1 --runs 20 -s -t 0.3

python check_model_output_single_turn.py "Run pwd" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "Check disk usage with df -h" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "Find all .py files in the current directory" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "Write a Python script that prints hello to /tmp/hello.py" 1 --runs 20 -s -t 0.3

python check_model_output_single_turn.py "List all markdown files" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "What is TEE?" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "What is Confidential Compute?" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "What is TDX?" 1 --runs 20 -s -t 0.3
python check_model_output_single_turn.py "Read all the Python files in cvm/ and summarize them" 1 --runs 20 -s -t 0.3


```